### Set Up

In [17]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [18]:
import numpy as np
import keras
import torch
import tensorflow as tf


In [19]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


### Load dataset

In [20]:
# Downloading an abbreviated collection of Shakespeare’s work
filename = keras.utils.get_file(origin=("https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"),)
shakespeare = open(filename, "r").read()
print(shakespeare[:250]), len(shakespeare)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



(None, 1115394)

### Data preprocess

In [21]:
# Splitting text into chunks for language model training
sequence_length = 100
def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]
features = list(split_input(shakespeare[:-1], sequence_length))
labels = list(split_input(shakespeare[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

In [22]:
features[:1], features[-1:]

(['First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'],
 ["\nNoble Sebastian,\nThou let'st thy fortune sleep--die, rather; wink'st\nWhiles thou art waking."])

In [23]:
labels[:1], labels[-1:]

(['irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '],
 ["Noble Sebastian,\nThou let'st thy fortune sleep--die, rather; wink'st\nWhiles thou art waking.\n"])

In [24]:
x, y = next(dataset.as_numpy_iterator())
x, y

(b'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou',
 b'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou ')

In [25]:
# Learning a character-level vocabulary with the TextVectorization layer
tokenizer = keras.layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

In [26]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

67

In [27]:
dataset = dataset.map(lambda features, labels: (tokenizer(features), tokenizer(labels)), num_parallel_calls=8,)
training_data = dataset.shuffle(10_000).batch(64).cache()

### Model

In [28]:
# Building a miniature language model
embedding_dim = 256
hidden_dim = 1024
inputs = keras.layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = keras.layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = keras.layers.GRU(hidden_dim, return_sequences=True)(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)          │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 100, 256)       │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 100, 1024)      │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 1024)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100, 67)        │        68,675 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

### Training

In [29]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="loss",
    restore_best_weights=True,
    patience=2,
)

In [30]:
# Training a miniature language mode
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=30, callbacks=[early_stopping])

Epoch 1/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - loss: 2.6848 - sparse_categorical_accuracy: 0.2779
Epoch 2/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - loss: 2.0010 - sparse_categorical_accuracy: 0.4145
Epoch 3/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 69ms/step - loss: 1.7477 - sparse_categorical_accuracy: 0.4810
Epoch 4/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 61ms/step - loss: 1.6090 - sparse_categorical_accuracy: 0.5179
Epoch 5/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - loss: 1.5205 - sparse_categorical_accuracy: 0.5415
Epoch 6/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - loss: 1.4569 - sparse_categorical_accuracy: 0.5579
Epoch 7/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - loss: 1.4079 - sparse_categorical_accuracy: 0.5706
Epoch 8/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - loss: 1.3663 - sparse_categorical_accuracy: 0.5815
Epoch 9/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - loss: 1.3289 - sparse_categorical_accuracy: 0.5912
Epoch 10/20
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s

### Inference

In [31]:
# Modifying the language model for autoregressive inference
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")
x = keras.layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = keras.layers.GRU(hidden_dim, return_state=True)(x, initial_state=input_state)
outputs = keras.layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(inputs=(inputs, input_state), outputs=(outputs, output_state), )
generation_model.set_weights(model.get_weights())
generation_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 256)    │     17,152 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state (InputLayer)  │ (None, 1024)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ [(None, 1024),    │  3,938,304 │ embedding_2[0][0… │
│                     │ (None, 1024)]     │            │ state[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 67)        │     68,675 │ gru_2[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,024,131 (15.35 MB)

 Trainable params: 4,024,131 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))
prompt = """Hi"""

In [33]:
# Using a fixed prompt to compute a language model’s starting state
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)
    print(predictions, state)

[[1.23474993e-05 4.83144795e-05 1.04850088e-03 5.93812048e-01
  3.28091810e-05 5.09668179e-02 8.28549489e-02 5.23609400e-04
  1.27682695e-04 1.15568298e-04 4.60708361e-05 9.58177168e-03
  1.45364820e-03 2.36758697e-05 7.80029859e-06 2.38414085e-03
  4.41943121e-05 1.98676996e-03 1.32357306e-03 2.19542359e-04
  1.42662830e-05 1.28142656e-05 3.86300417e-06 9.11388732e-03
  1.15414268e-05 3.42031308e-05 1.33860214e-02 1.36972975e-03
  5.74834235e-02 2.23284696e-05 3.55450584e-06 1.49808312e-03
  3.28028138e-04 7.40795583e-02 7.47607574e-02 6.36629644e-04
  4.21529607e-04 7.48180100e-05 1.09816581e-04 5.73333760e-04
  7.47213606e-04 3.00254044e-03 3.78764956e-03 3.20036488e-04
  8.17498250e-04 1.71962415e-03 6.19692553e-04 4.51038504e-04
  4.56372218e-04 2.47756747e-04 5.47930249e-05 3.91406094e-04
  4.57426440e-03 7.33149820e-04 3.63397303e-05 6.15873025e-04
  4.24352365e-05 2.89244399e-05 1.38736123e-05 1.48430277e-04
  1.95072760e-04 1.77581562e-04 1.15321025e-04 1.93576361e-05
  6.2947

In [34]:
# Predicting with the language model a token at a time
generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [35]:
generated_ids

[8,
 2,
 8,
 19,
 11,
 20,
 4,
 2,
 6,
 10,
 14,
 2,
 14,
 11,
 8,
 25,
 5,
 8,
 11,
 4,
 11,
 5,
 10,
 18,
 2,
 6,
 10,
 14,
 2,
 7,
 3,
 9,
 2,
 24,
 9,
 5,
 4,
 7,
 3,
 9,
 8,
 18,
 12,
 28,
 10,
 14,
 2,
 4,
 7,
 3,
 9,
 3,
 20,
 5,
 9,
 3,
 2,
 7,
 6,
 29,
 3,
 2,
 23,
 2,
 8,
 3,
 10,
 4,
 2,
 20,
 5,
 9,
 2,
 4,
 7,
 3,
 11,
 9,
 2,
 20,
 9,
 11,
 3,
 10,
 14,
 8,
 27,
 12,
 12,
 54,
 23,
 35,
 47,
 2,
 36,
 23,
 39,
 43,
 28,
 36,
 49,
 2,
 23,
 23,
 23,
 26,
 12,
 37,
 11,
 9,
 18,
 2,
 23,
 2,
 7,
 6,
 29,
 3,
 2,
 8,
 5,
 16,
 3,
 2,
 8,
 25,
 3,
 3,
 21,
 7,
 2,
 5,
 20,
 2,
 21,
 7,
 6,
 9,
 11,
 4,
 17,
 18,
 2,
 6,
 10,
 14,
 2,
 7,
 11,
 8,
 2,
 8,
 5,
 10,
 22,
 8,
 32,
 2,
 14,
 3,
 6,
 4,
 7,
 27,
 12,
 12,
 54,
 23,
 35,
 47,
 2,
 33,
 49,
 41,
 28,
 36,
 49,
 2,
 23,
 55,
 26,
 12,
 37,
 3,
 3,
 2,
 19,
 7,
 3,
 9,
 3,
 2,
 8,
 7,
 3,
 2,
 8,
 4,
 6,
 10,
 14,
 8,
 2,
 24,
 17,
 2,
 6,
 2,
 16,
 6,
 10,
 2,
 4,
 7,
 6,
 4,
 2,
 14,
 5,
 2,
 4,
 7,
 3,
 2,
 22,
 9,


In [36]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)

His swift and disposition, and her brothers,
And therefore have I sent for their friends.

KING RICHARD III:
Sir, I have some speech of charity, and his songs' death.

KING EDWARD IV:
See where she stands by a man that do the grace of me;
And therefore
